#### to run this file make sure to download the dataset from this link https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

### loading the data

In [2]:
reviews = []
lables = []

positive_reviews = os.path.join("aclImdb/train/pos")
negative_reviews = os.path.join("aclImdb/train/neg")
for file in os.listdir(positive_reviews):
    with open(os.path.join(positive_reviews, file), 'r') as f:
        reviews.append((f.read()).lower())
        lables.append(1)
for file in os.listdir(negative_reviews):
    with open(os.path.join(negative_reviews, file), 'r') as f:
        reviews.append((f.read()).lower())
        lables.append(0)
train_data = pd.DataFrame({"review": reviews, "label": lables})
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   review  25000 non-null  object
 1   label   25000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 390.8+ KB


### preprocessign the text

In [3]:
train_data["review"] = train_data["review"].apply(lambda x: re.sub(r"[^\w ]", " ", x))
train_data["review"] = train_data["review"].apply(lambda x: x.split())
train_data.head()

,review,label
0,"[bromwell, high, is, a, cartoon, comedy, it, r...",1
1,"[homelessness, or, houselessness, as, george, ...",1
2,"[brilliant, over, acting, by, lesley, ann, war...",1
3,"[this, is, easily, the, most, underrated, film...",1
4,"[this, is, not, the, typical, mel, brooks, fil...",1


### creating the vocab

In [4]:
vocab = {}
index = 0
for review in train_data["review"]:
    for word in review:
        if word not in vocab:
            vocab[word] = index
            index += 1
print(vocab)

{'bromwell': 0, 'high': 1, 'is': 2, 'a': 3, 'cartoon': 4, 'comedy': 5, 'it': 6, 'ran': 7, 'at': 8, 'the': 9, 'same': 10, 'time': 11, 'as': 12, 'some': 13, 'other': 14, 'programs': 15, 'about': 16, 'school': 17, 'life': 18, 'such': 19, 'teachers': 20, 'my': 21, '35': 22, 'years': 23, 'in': 24, 'teaching': 25, 'profession': 26, 'lead': 27, 'me': 28, 'to': 29, 'believe': 30, 'that': 31, 's': 32, 'satire': 33, 'much': 34, 'closer': 35, 'reality': 36, 'than': 37, 'scramble': 38, 'survive': 39, 'financially': 40, 'insightful': 41, 'students': 42, 'who': 43, 'can': 44, 'see': 45, 'right': 46, 'through': 47, 'their': 48, 'pathetic': 49, 'pomp': 50, 'pettiness': 51, 'of': 52, 'whole': 53, 'situation': 54, 'all': 55, 'remind': 56, 'schools': 57, 'i': 58, 'knew': 59, 'and': 60, 'when': 61, 'saw': 62, 'episode': 63, 'which': 64, 'student': 65, 'repeatedly': 66, 'tried': 67, 'burn': 68, 'down': 69, 'immediately': 70, 'recalled': 71, 'classic': 72, 'line': 73, 'inspector': 74, 'm': 75, 'here': 76, '

### turning the reviews into bag of words representation (using sparse vectorization method)

In [10]:
def create_bow(review):
    bow = {}
    for word in review:
        if word in vocab:
            if vocab[word] in bow:
                bow[vocab[word]] += 1
            else:
                bow[vocab[word]] = 1
    return bow

train_data["review_bow"] = train_data["review"].apply(create_bow)
train_data.head()

,review,label,review_bow
0,"[bromwell, high, is, a, cartoon, comedy, it, r...",1,"{0: 4, 1: 5, 2: 4, 3: 4, 4: 1, 5: 1, 6: 2, 7: ..."
1,"[homelessness, or, houselessness, as, george, ...",1,"{92: 1, 93: 8, 94: 1, 12: 5, 95: 1, 96: 1, 97:..."
2,"[brilliant, over, acting, by, lesley, ann, war...",1,"{292: 1, 293: 1, 294: 1, 208: 1, 219: 1, 220: ..."
3,"[this, is, easily, the, most, underrated, film...",1,"{287: 2, 2: 4, 361: 1, 9: 6, 121: 2, 362: 1, 2..."
4,"[this, is, not, the, typical, mel, brooks, fil...",1,"{287: 2, 2: 3, 241: 1, 9: 8, 410: 1, 168: 1, 1..."


### computing class priors

In [6]:
priors = train_data["label"].value_counts(normalize=True).to_dict()
print(priors)

{1: 0.5, 0: 0.5}


### computing likelihoods for word using laplace smoothing

In [7]:
positive_train_data = train_data[train_data["label"] == 1]
negative_train_data = train_data[train_data["label"] == 0]
likelihoods = {}
for word, index in vocab.items():
    likelihoods[index] = {
        1: (positive_train_data["review_bow"].apply(lambda x: x.get(index, 0)).sum() + 1) / (positive_train_data["review_bow"].apply(lambda x: sum(x.values())).sum() + len(vocab)),
        0: (negative_train_data["review_bow"].apply(lambda x: x.get(index, 0)).sum() + 1) / (negative_train_data["review_bow"].apply(lambda x: sum(x.values())).sum() + len(vocab))
    }

### loading test data

In [12]:
reviews = []
lables = []

positive_reviews = os.path.join("aclImdb/test/pos")
negative_reviews = os.path.join("aclImdb/test/neg")
for file in os.listdir(positive_reviews):
    with open(os.path.join(positive_reviews, file), 'r') as f:
        reviews.append((f.read()).lower())
        lables.append(1)
for file in os.listdir(negative_reviews):
    with open(os.path.join(negative_reviews, file), 'r') as f:
        reviews.append((f.read()).lower())
        lables.append(0)
test_data = pd.DataFrame({"review": reviews, "label": lables})
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   review  25000 non-null  object
 1   label   25000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 390.8+ KB


### preprocessing test data

In [13]:
test_data["review"] = test_data["review"].apply(lambda x: re.sub(r"[^\w ]", " ", x))
test_data["review"] = test_data["review"].apply(lambda x: x.split())
test_data["review_bow"] = test_data["review"].apply(create_bow)
test_data.head()

,review,label,review_bow
0,"[i, went, and, saw, this, movie, last, night, ...",1,"{58: 7, 2337: 1, 60: 4, 62: 2, 287: 3, 355: 4,..."
1,"[actor, turned, director, bill, paxton, follow...",1,"{441: 1, 3382: 1, 407: 1, 2184: 1, 2185: 3, 11..."
2,"[as, a, recreational, golfer, with, some, know...",1,"{12: 3, 3: 3, 38793: 1, 70988: 1, 179: 2, 13: ..."
3,"[i, saw, this, film, in, a, sneak, preview, an...",1,"{58: 3, 62: 1, 287: 5, 288: 3, 24: 1, 3: 6, 10..."
4,"[bill, paxton, has, taken, the, true, story, o...",1,"{2184: 1, 2185: 1, 98: 1, 929: 1, 9: 20, 561: ..."


### generating predictions

In [ ]:
predictions = []
for _, row in test_data.iterrows():
    positive_prob = priors[1]
    negative_prob = priors[0]
    for index, count in row["review_bow"].items():
        positive_prob *= likelihoods[index][1] ** count
        negative_prob *= likelihoods[index][0] ** count
    if positive_prob > negative_prob:
        predictions.append(1)
    else:
        predictions.append(0)

In [16]:
model_accuracy = np.mean(np.array(predictions) == test_data["label"])
print(f"Accuracy: {model_accuracy}")

Accuracy: 0.55072


In [18]:
predictions = []
for _, row in test_data.iterrows():
    log_positive_prob = np.log(priors[1])
    log_negative_prob = np.log(priors[0])
    for index, count in row["review_bow"].items():
        log_positive_prob += count * np.log(likelihoods[index][1])
        log_negative_prob += count * np.log(likelihoods[index][0])
    if log_positive_prob > log_negative_prob:
        predictions.append(1)
    else:
        predictions.append(0)

In [19]:
model_accuracy = np.mean(np.array(predictions) == test_data["label"])
print(f"Accuracy: {model_accuracy}")

Accuracy: 0.81016
